# 2. Build survey targets


**Input**
- ANES 2024, CCES 2024 and GSS 2024 raw files in `original_data/`
- `updated_belief_features.json` -- the 40 belief questions and their option lists
- `feature_extraction/cleaned_all_result_7d.csv` -- only for the 52 window start dates

**Output** (`generated_data/target_df_7d/`)
- `basic_anes_target_df_7d.csv`, `basic_cces_target_df_7d.csv`, `basic_gss_target_df_7d.csv`
- `target_map_7d.csv` -- one row per question x survey: source variable, mapping, K, flags

**Which targets**
- Every survey listed for a question in `belief_question_cleanup/features_no_extra_columns.xlsx` (40 wording-source targets + 28 others on 25 questions). The table is copied into section 1 so the mapping can be read and checked here.
- The question wording follows GSS first, then ANES, then CCES, so the wording-source survey is the one named by the belief id (`V24*` = ANES, `CC24_*` = CCES, otherwise GSS). `wording_source` marks it.



In [1]:
import json
import os

import numpy as np
import pandas as pd

from belief_recodes_7d import RECODES, n_categories, recode_value

ORIGINAL_DIR = "../../Dropbox/is_culture_king/is_culture_king_Yuyue/original_data"
DATA_DIR = "../../Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data"
ANES_CSV = f"{ORIGINAL_DIR}/anes_timeseries_2024_csv_20250808/anes_timeseries_2024_csv_20250808.csv"
CCES_CSV = f"{ORIGINAL_DIR}/CCES_2024/CCES24_Common_OUTPUT_vv_topost_final.csv"
GSS_CSV = f"{ORIGINAL_DIR}/gss2024.csv"
FEATURES_JSON = f"{DATA_DIR}/updated_belief_features.json"
WINDOWS_CSV = f"{DATA_DIR}/feature_extraction/cleaned_all_result_7d.csv"

OUT_DIR = f"{DATA_DIR}/target_df_7d"
BENCHMARK = {"ANES": "ANES2024", "CCES": "CCES2024", "GSS": "gss"}  # names used by the biweekly targets

## 1. Targets

`map` takes a survey code to a value on the comparison scale; codes it doesn't list (refused, don't know, "Not sure", neutral options with no counterpart) become missing. `map=None` means the survey's valid codes already are our option positions. The codes were checked against the ANES / GSS codebooks and the CCES questionnaires.

`llm_map` is set only when the target can't sit on our full scale: it maps our recoded answer (1..K) to the target's scale, and options it leaves out are dropped from the comparison.

In [2]:
with open(FEATURES_JSON, "r", encoding="utf-8") as f:
    belief_features = json.load(f)["beliefs"]
options = {f["id"]: f["options"] for f in belief_features}

def wording_source(belief_id):
    if belief_id.startswith("V24"):
        return "ANES"
    if belief_id.startswith("CC24_"):
        return "CCES"
    return "GSS"

# the id is the source variable, except where the data column is named differently
SOURCE_VAR = {"CC24_443d": "CC24_443_4"}

TARGETS = [
    {"id": f["id"], "survey": wording_source(f["id"]),
     "var": SOURCE_VAR.get(f["id"], f["id"].lower() if wording_source(f["id"]) == "GSS" else f["id"])}
    for f in belief_features
]

NOT_SURE_4 = {1: 1, 2: 2, 3: 3, 4: 4}        # CCES approval, 5 = "Not sure"
YES_NO = {1: 1, 2: 2}                        # CCES support / oppose
FAVOR_7_TO_BINARY = {1: 1, 2: 1, 3: 1, 5: 2, 6: 2, 7: 2}  # our 7-point favor / oppose -> support / oppose, neutral dropped

TARGETS += [
    # same scale and direction, only the wording differs
    {"id": "LETIN1A", "survey": "ANES", "var": "V242227"},
    {"id": "IMMCRIME1", "survey": "ANES", "var": "V242453"},
    {"id": "WRKWAYUP", "survey": "ANES", "var": "V242300"},
    {"id": "WRKWAYUP", "survey": "CCES", "var": "CC24_441a"},
    {"id": "V242301", "survey": "CCES", "var": "CC24_441b"},
    {"id": "V241608", "survey": "CCES", "var": "CC24_440c"},
    {"id": "AFFRMACT", "survey": "ANES", "var": "V242241x"},
    {"id": "EQWLTH", "survey": "ANES", "var": "V242253x"},
    {"id": "V241294x", "survey": "CCES", "var": "CC24_301", "map": {1: 1, 2: 2, 3: 3, 4: 4, 5: 5}},  # 6 = "Not sure"
    {"id": "V241137x", "survey": "CCES", "var": "CC24_312a", "map": NOT_SURE_4},
    {"id": "V241129x", "survey": "CCES", "var": "CC24_312c", "map": NOT_SURE_4},
    {"id": "V241133x", "survey": "CCES", "var": "CC24_312d", "map": NOT_SURE_4},
    {"id": "SATDEMOC", "survey": "ANES", "var": "V242439", "map": {1: 1, 2: 2, 4: 3, 5: 4}},  # ANES has no code 3

    # merged with the biweekly conventions
    {"id": "CAPPUN", "survey": "ANES", "var": "V241308x", "map": {1: 1, 2: 1, 3: 2, 4: 2},
     "note": "favor / oppose strongly + not strongly merged"},
    {"id": "ABANY", "survey": "CCES", "var": "CC24_324a", "map": YES_NO,
     "note": "support 'always allow ... as a matter of choice' = yes"},
    {"id": "MARSAME", "survey": "ANES", "var": "V241385x", "map": {1: 1, 2: 2, 3: 2, 4: 3, 5: 4, 6: 4, 7: 5},
     "note": "7-point favor / oppose; moderate + little merged"},
    {"id": "FEFAM", "survey": "ANES", "var": "V242279x", "map": {1: 1, 2: 2, 3: 2, 5: 3, 6: 3, 7: 4},
     "note": "better / worse for the family; 4 'makes no difference' (~49%) dropped"},
    {"id": "IMMFATE", "survey": "ANES", "var": "V241386", "map": {1: 3, 2: 2, 3: 1, 4: 1},
     "note": "send back -> deported, guest worker -> permanent resident, both stay-and-qualify -> citizens"},
    {"id": "CONLABORY", "survey": "ANES", "var": "V242140", "map": "thermometer",
     "note": "0-100 thermometer cut into equal thirds, reversed"},
    {"id": "BIASNEWS", "survey": "ANES", "var": "V241335",
     "note": "trust in news media: 1 'none' ~ 'strongly agree it is biased'; different construct"},
    {"id": "TEMPGEN1", "survey": "ANES", "var": "V242174",
     "note": "importance of climate change rather than its danger; different construct"},

    # weak matches, kept but flagged
    {"id": "IMMJOBS1", "survey": "ANES", "var": "V242228", "map": {1: 1, 2: 2, 3: 4, 4: 5}, "low": True,
     "note": "how likely immigration takes jobs; 'somewhat likely' counted as 'disagree', as in the biweekly targets"},
    {"id": "V241229", "survey": "CCES", "var": "CC24_423", "map": {1: 1, 2: 2, 3: 3, 8: 4}, "low": True,
     "llm_map": {1: 1, 2: 2, 3: 3, 4: 3, 5: 4},
     "note": "how much trust (8 = none at all); our 'about half' and 'some of the time' merged"},
    {"id": "ABANY", "survey": "ANES", "var": "V241248", "map": {1: 1, 2: 1, 3: 1, 5: 2, 6: 2, 7: 2}, "low": True,
     "note": "7-point always permitted .. never permitted; 4 and 99 'haven't thought' dropped"},
    {"id": "MARSAME", "survey": "CCES", "var": "CC24_340c", "map": YES_NO, "low": True,
     "llm_map": {1: 1, 2: 1, 4: 2, 5: 2},
     "note": "bill recognising same-sex and interracial marriages; our neutral dropped"},
    {"id": "V242331x", "survey": "CCES", "var": "CC24_321a", "map": YES_NO, "low": True,
     "llm_map": FAVOR_7_TO_BINARY, "note": "binary support / oppose; our neutral dropped"},
    {"id": "V242328x", "survey": "CCES", "var": "CC24_321c", "map": YES_NO, "low": True,
     "llm_map": FAVOR_7_TO_BINARY, "note": "binary support / oppose; our neutral dropped"},
    {"id": "V241395x", "survey": "CCES", "var": "CC24_323c", "map": YES_NO, "low": True,
     "llm_map": FAVOR_7_TO_BINARY, "note": "binary support / oppose; our neutral dropped"},
]

def wave(survey, var):
    if survey == "ANES":
        return {"V241": "pre", "V242": "post"}[var[:4]]
    if survey == "CCES":
        return {"CC24_3": "pre", "CC24_4": "post"}[var[:6]]
    return "single"

target_map = pd.DataFrame([{
    "id": t["id"], "question_id": f"Belief_{t['id']}_Symbol", "survey": t["survey"], "benchmark": BENCHMARK[t["survey"]],
    "var": t["var"], "wave": wave(t["survey"], t["var"]), "wording_source": t["survey"] == wording_source(t["id"]),
    "confidence_low": t.get("low", False), "map": t.get("map"), "llm_map": t.get("llm_map"), "note": t.get("note", ""),
} for t in TARGETS])

assert not target_map.duplicated(["id", "survey"]).any()
assert target_map["wording_source"].sum() == len(belief_features) == target_map["id"].nunique()
# the shared recodes only work on our full scale
assert not target_map.loc[target_map["id"].isin(RECODES), "llm_map"].notna().any()
print(f"{len(target_map)} targets: {target_map['wording_source'].sum()} wording source, "
      f"{(~target_map['wording_source']).sum()} others ({target_map['confidence_low'].sum()} low confidence)")
target_map.groupby(["survey", "wave"]).size().unstack(fill_value=0)

68 targets: 40 wording source, 28 others (7 low confidence)


wave,post,pre,single
survey,,,
ANES,15,15,0
CCES,7,10,0
GSS,0,0,21


## 2. Load surveys

Only the columns used here. GSS variables are lowercase in `gss2024.csv`.

In [3]:
DEMO_VARS = {
    "ANES": ["V240001", "V241550", "V241458x", "V241566x", "V241501x", "V241465x", "V241106x", "V242058x",
             "V243050", "V243054", "V243060"],
    "CCES": ["caseid", "gender4", "birthyr", "faminc_new", "race", "educ", "presvote20post", "inputstate",
             "starttime", "starttime_post"],
    "GSS": ["id", "sex", "age", "income16", "racecen1", "racecen2", "degree", "vote20", "pres20", "dateintv"],
}
paths = {"ANES": ANES_CSV, "CCES": CCES_CSV, "GSS": GSS_CSV}

surveys = {}
for survey, path in paths.items():
    belief_vars = target_map.loc[target_map["survey"] == survey, "var"].tolist()
    surveys[survey] = pd.read_csv(path, usecols=DEMO_VARS[survey] + belief_vars, low_memory=False)
    print(f"{survey}: {len(surveys[survey]):,} respondents, {len(belief_vars)} belief variables")

ANES: 5,521 respondents, 30 belief variables


CCES: 60,000 respondents, 17 belief variables


GSS: 3,309 respondents, 21 belief variables


## 3. Demographics, state and interview date

Same groups and labels as the biweekly targets: sex Male / Female, age 18-34 / 35-54 / 55+, income <50k / 50-100k / 100k+, White / Non-white, College / Non-college, 2020 vote StayHome / Democratic / Republican / Other. State is the full name (none in the public GSS), plus `state_abbr`.

Fixed against the biweekly notebook:
- Age: `pd.cut` there used right-closed bins, so 18-year-olds were dropped and ages 35 and 55 fell into the younger group.
- College = bachelor's or higher in all three. The biweekly code counted GSS associate degrees and CCES 2-year degrees as college but ANES associate degrees as non-college.
- GSS: people who didn't vote in 2020 (`vote20` = 2) were missing, because `pres20` is only asked of voters. They are now StayHome; ineligible stays missing.
- ANES: paper (PAPI) respondents have no web / phone start date (`V243050`); their completion date `V243054` is used instead, when it falls inside the pre-election field period (it is written in by the respondent, and a few are clearly wrong).

GSS income code 22 is $90,000-109,999 and stays in 100k+, as before.

In [4]:
FIPS = {
    1: ("Alabama", "AL"), 2: ("Alaska", "AK"), 4: ("Arizona", "AZ"), 5: ("Arkansas", "AR"), 6: ("California", "CA"),
    8: ("Colorado", "CO"), 9: ("Connecticut", "CT"), 10: ("Delaware", "DE"), 11: ("District of Columbia", "DC"),
    12: ("Florida", "FL"), 13: ("Georgia", "GA"), 15: ("Hawaii", "HI"), 16: ("Idaho", "ID"), 17: ("Illinois", "IL"),
    18: ("Indiana", "IN"), 19: ("Iowa", "IA"), 20: ("Kansas", "KS"), 21: ("Kentucky", "KY"), 22: ("Louisiana", "LA"),
    23: ("Maine", "ME"), 24: ("Maryland", "MD"), 25: ("Massachusetts", "MA"), 26: ("Michigan", "MI"),
    27: ("Minnesota", "MN"), 28: ("Mississippi", "MS"), 29: ("Missouri", "MO"), 30: ("Montana", "MT"),
    31: ("Nebraska", "NE"), 32: ("Nevada", "NV"), 33: ("New Hampshire", "NH"), 34: ("New Jersey", "NJ"),
    35: ("New Mexico", "NM"), 36: ("New York", "NY"), 37: ("North Carolina", "NC"), 38: ("North Dakota", "ND"),
    39: ("Ohio", "OH"), 40: ("Oklahoma", "OK"), 41: ("Oregon", "OR"), 42: ("Pennsylvania", "PA"),
    44: ("Rhode Island", "RI"), 45: ("South Carolina", "SC"), 46: ("South Dakota", "SD"), 47: ("Tennessee", "TN"),
    48: ("Texas", "TX"), 49: ("Utah", "UT"), 50: ("Vermont", "VT"), 51: ("Virginia", "VA"), 53: ("Washington", "WA"),
    54: ("West Virginia", "WV"), 55: ("Wisconsin", "WI"), 56: ("Wyoming", "WY"),
}

def age_group(age):
    # left-closed, so 18 -> 18-34 and 35 -> 35-54
    return pd.cut(age, bins=[18, 35, 55, np.inf], labels=["18-34", "35-54", "55+"], right=False).astype(object)

def income_group(code, under_50k, under_100k, top):
    return pd.Series(np.select(
        [code.between(1, under_50k), code.between(under_50k + 1, under_100k), code.between(under_100k + 1, top)],
        ["<50k", "50-100k", "100k+"], default=None), index=code.index)

def yyyymmdd(col):
    return pd.to_datetime(pd.to_numeric(col, errors="coerce").astype("Int64").astype("string"),
                          format="%Y%m%d", errors="coerce")

def add_state(df, fips):
    df["state"] = fips.map(lambda v: FIPS.get(v, (None, None))[0])
    df["state_abbr"] = fips.map(lambda v: FIPS.get(v, (None, None))[1])

anes, cces, gss = surveys["ANES"], surveys["CCES"], surveys["GSS"]

anes["respondent_id"] = anes["V240001"]
anes["sex"] = anes["V241550"].map({1: "Male", 2: "Female"})
anes["age"] = age_group(anes["V241458x"].where(anes["V241458x"] >= 18))
anes["income"] = income_group(anes["V241566x"], 14, 22, 28)  # 14 = $45,000-49,999, 22 = $90,000-99,999
anes["eth"] = anes["V241501x"].map({1: "White", 2: "Non-white", 3: "Non-white", 4: "Non-white", 5: "Non-white", 6: "Non-white"})
anes["education"] = anes["V241465x"].map({1: "Non-college", 2: "Non-college", 3: "Non-college", 4: "College", 5: "College"})
anes["vote"] = anes["V241106x"].map({1: "StayHome", 2: "Democratic", 3: "Republican", 4: "Other"})
add_state(anes, anes["V242058x"])  # state of registration, as before
web_date = yyyymmdd(anes["V243050"])
# PAPI completion dates are written in by the respondent; keep only those inside the pre-election field period
papi_date = yyyymmdd(anes["V243054"])
anes["interview_date"] = web_date.fillna(papi_date.where(papi_date.between(web_date.min(), web_date.max())))
anes["post_interview_date"] = yyyymmdd(anes["V243060"])

cces["respondent_id"] = cces["caseid"]
cces["sex"] = cces["gender4"].map({1: "Male", 2: "Female"})
cces["age"] = age_group(2024 - cces["birthyr"])
cces["income"] = income_group(cces["faminc_new"], 5, 9, 16)  # 5 = $40,000-49,999, 9 = $80,000-99,999; 97 = refused
cces["eth"] = np.where(cces["race"] == 1, "White", "Non-white")
cces["education"] = cces["educ"].map({1: "Non-college", 2: "Non-college", 3: "Non-college", 4: "Non-college", 5: "College", 6: "College"})
cces["vote"] = cces["presvote20post"].map({1: "Democratic", 2: "Republican", 3: "Other", 4: "Other", 5: "Other", 6: "StayHome"})
add_state(cces, cces["inputstate"])
cces["interview_date"] = pd.to_datetime(cces["starttime"]).dt.normalize()
cces["post_interview_date"] = pd.to_datetime(cces["starttime_post"]).dt.normalize()

gss["respondent_id"] = gss["id"]
gss["sex"] = gss["sex"].map({1: "Male", 2: "Female"})
gss["age"] = age_group(gss["age"])
gss["income"] = income_group(gss["income16"], 18, 21, 26)  # 18 = $40,000-49,999, 21 = $75,000-89,999
white_only = gss["racecen1"].eq(1) & gss["racecen2"].isna()
gss["eth"] = np.where(gss["racecen1"].isna(), None, np.where(white_only, "White", "Non-white"))
gss["education"] = gss["degree"].map({0: "Non-college", 1: "Non-college", 2: "Non-college", 3: "College", 4: "College"})
gss["vote"] = gss["pres20"].map({1: "Democratic", 2: "Republican", 3: "Other", 4: "StayHome"})
gss.loc[gss["vote20"].eq(2), "vote"] = "StayHome"
gss["state"] = gss["state_abbr"] = None
# dateintv is MMDD; all 2024 interviews ran April-December
gss["interview_date"] = pd.to_datetime("2024" + gss["dateintv"].astype(int).astype(str).str.zfill(4), format="%Y%m%d", errors="coerce")
gss["post_interview_date"] = pd.NaT

DEMO_COLS = ["sex", "age", "income", "eth", "education", "vote", "state", "state_abbr"]
for survey, df in surveys.items():
    missing = df[DEMO_COLS[:-2] + ["interview_date"]].isna().mean().round(3).to_dict()
    print(f"{survey} share missing: {missing}")

ANES share missing: {'sex': 0.008, 'age': 0.051, 'income': 0.101, 'eth': 0.012, 'education': 0.01, 'vote': 0.009, 'interview_date': 0.002}
CCES share missing: {'sex': 0.009, 'age': 0.0, 'income': 0.086, 'eth': 0.0, 'education': 0.0, 'vote': 0.001, 'interview_date': 0.0}
GSS share missing: {'sex': 0.006, 'age': 0.031, 'income': 0.114, 'eth': 0.02, 'education': 0.001, 'vote': 0.167, 'interview_date': 0.0}


In [5]:
# group shares per survey, to eyeball against the biweekly targets and the codebooks
pd.concat({
    survey: pd.concat({col: df[col].value_counts(normalize=True) for col in DEMO_COLS[:-2]})
    for survey, df in surveys.items()
}, axis=1).round(3)

ANES   CCES    GSS
sex       Female       0.535  0.538  0.554
          Male         0.465  0.462  0.446
age       55+          0.492  0.519  0.436
          35-54        0.328  0.283  0.331
          18-34        0.181  0.198  0.232
income    100k+        0.394  0.255  0.342
          <50k         0.326  0.428  0.406
          50-100k      0.280  0.317  0.252
eth       White        0.724  0.691  0.684
          Non-white    0.276  0.309  0.316
education Non-college  0.548  0.646  0.639
          College      0.452  0.354  0.361
vote      Democratic   0.444  0.449  0.431
          Republican   0.333  0.324  0.317
          StayHome     0.211  0.199  0.233
          Other        0.012  0.028  0.019

## 4. Time windows

The 52 window starts of the extraction. A respondent falls in `[start , start + 7 days)` of their interview date, labelled like the biweekly `time_window_bin`. Interviews after the last window (and GSS interviews after Election Day) get no window.

In [6]:
starts = pd.read_csv(WINDOWS_CSV, usecols=["time_window_start"])["time_window_start"].unique()
starts = pd.DatetimeIndex(pd.to_datetime(starts).tz_localize(None)).sort_values()
assert len(starts) == 52 and (starts[1:] - starts[:-1] == pd.Timedelta(days=7)).all()

edges = starts.append(pd.DatetimeIndex([starts[-1] + pd.Timedelta(days=7)]))
window_labels = [f"[{a.date()} , {b.date()})" for a, b in zip(edges[:-1], edges[1:])]

for survey, df in surveys.items():
    df["time_window_bin"] = pd.cut(df["interview_date"], bins=edges, labels=window_labels, right=False).astype(object)
    in_window = df["time_window_bin"].notna()
    print(f"{survey}: {in_window.sum():,} of {len(df):,} respondents in a window, "
          f"interviews {df['interview_date'].min().date()} to {df['interview_date'].max().date()}, "
          f"{df.loc[in_window, 'time_window_bin'].nunique()} windows")

ANES: 5,436 of 5,521 respondents in a window, interviews 2024-08-03 to 2024-11-05, 14 windows
CCES: 59,990 of 60,000 respondents in a window, interviews 2024-10-01 to 2024-11-04, 5 windows
GSS: 2,983 of 3,309 respondents in a window, interviews 2024-04-05 to 2024-12-02, 31 windows


## 5. Belief values

In [7]:
def target_values(t, raw):
    raw = pd.to_numeric(raw, errors="coerce")
    if t["map"] is None:
        n = len(options[t["id"]])
        valid = raw.where(raw.isin(range(1, n + 1)))
        return valid.map(lambda v: recode_value(t["id"], v))
    if t["map"] == "thermometer":
        valid = raw.where(raw.between(0, 100))
        return pd.Series(np.select([valid < 100 / 3, valid < 200 / 3, valid <= 100], [3, 2, 1], default=np.nan), index=raw.index)
    return raw.map(t["map"])

def target_k(t):
    if t["llm_map"] is not None:
        return len(set(t["llm_map"].values()))
    return n_categories(t["id"], len(options[t["id"]]))

target_map["K"] = target_map.apply(target_k, axis=1)

parts = {survey: [] for survey in surveys}
summary = []
for _, t in target_map.iterrows():
    df = surveys[t["survey"]]
    values = target_values(t, df[t["var"]])
    assert values.dropna().isin(range(1, t["K"] + 1)).all(), (t["id"], t["survey"])
    assert values.notna().any(), (t["id"], t["survey"])

    part = df[["respondent_id"] + DEMO_COLS + ["time_window_bin", "interview_date", "post_interview_date"]].copy()
    part["belief_value"] = values
    part["question_id"] = t["question_id"]
    part["benchmark"] = t["benchmark"]
    part["wave"] = t["wave"]
    part["wording_source"] = t["wording_source"]
    part["confidence_low"] = t["confidence_low"]
    parts[t["survey"]].append(part[part["belief_value"].notna()])

    shares = values.value_counts(normalize=True).reindex(range(1, t["K"] + 1), fill_value=0)
    summary.append({"id": t["id"], "survey": t["survey"], "var": t["var"], "wave": t["wave"],
                    "wording_source": t["wording_source"], "low": t["confidence_low"], "K": t["K"],
                    "n": int(values.notna().sum()), "n_in_window": int((values.notna() & df["time_window_bin"].notna()).sum()),
                    "mean": values.mean(), "shares": " / ".join(f"{s:.2f}" for s in shares)})

summary = pd.DataFrame(summary)
pd.set_option("display.max_rows", 100, "display.width", 200)
summary.round(2)

,id,survey,var,wave,wording_source,low,K,n,n_in_window,mean,shares
0,LETIN1A,GSS,letin1a,single,True,False,5,3187,2870,3.55,0.07 / 0.11 / 0.32 / 0.19 / 0.30
1,IMMCRIME1,GSS,immcrime1,single,True,False,5,1545,1400,3.26,0.08 / 0.19 / 0.26 / 0.32 / 0.15
2,IMMJOBS1,GSS,immjobs1,single,True,False,5,1563,1414,3.33,0.07 / 0.20 / 0.22 / 0.36 / 0.15
3,IMMFATE,GSS,immfate,single,True,False,3,2170,1955,1.65,0.60 / 0.15 / 0.25
4,DIVLAW,GSS,divlaw,single,True,False,3,903,808,1.77,0.50 / 0.23 / 0.27
5,ABANY,GSS,abany,single,True,False,2,2140,1925,1.40,0.60 / 0.40
6,MARSAME,GSS,marsame,single,True,False,5,1127,1016,2.29,0.41 / 0.23 / 0.16 / 0.07 / 0.13
7,FEPOLV,GSS,fepolv,single,True,False,2,500,450,1.84,0.16 / 0.84
8,FEFAM,GSS,fefam,single,True,False,4,2162,1946,2.90,0.08 / 0.23 / 0.42 / 0.28
9,V242353x,ANES,V242353x,post,True,False,7,4692,4634,3.02,0.24 / 0.26 / 0.06 / 0.29 / 0.02 / 0.07 / 0.05


In [8]:
# raw code -> target value for every mapped target, to check the merges by eye
for _, t in target_map[target_map["map"].notna()].iterrows():
    raw = pd.to_numeric(surveys[t["survey"]][t["var"]], errors="coerce")
    values = target_values(t, raw)
    if t["map"] == "thermometer":
        check = values.groupby(values).agg(lambda s: f"{raw[s.index].min():.0f}-{raw[s.index].max():.0f}")
    else:
        check = pd.crosstab(raw, values.fillna(-1)).idxmax(axis=1).replace(-1, "missing")
    print(f"{t['id']:10} {t['survey']} {t['var']}: {check.to_dict()}")

V241294x   CCES CC24_301: {1.0: 1.0, 2.0: 2.0, 3.0: 3.0, 4.0: 4.0, 5.0: 5.0, 6.0: 'missing'}
V241137x   CCES CC24_312a: {1.0: 1.0, 2.0: 2.0, 3.0: 3.0, 4.0: 4.0, 5.0: 'missing'}
V241129x   CCES CC24_312c: {1.0: 1.0, 2.0: 2.0, 3.0: 3.0, 4.0: 4.0, 5.0: 'missing'}
V241133x   CCES CC24_312d: {1.0: 1.0, 2.0: 2.0, 3.0: 3.0, 4.0: 4.0, 5.0: 'missing'}
SATDEMOC   ANES V242439: {-9: 'missing', -8: 'missing', -7: 'missing', -6: 'missing', -5: 'missing', -1: 'missing', 1: 1.0, 2: 2.0, 4: 3.0, 5: 4.0}
CAPPUN     ANES V241308x: {-2: 'missing', -1: 'missing', 1: 1.0, 2: 1.0, 3: 2.0, 4: 2.0}
ABANY      CCES CC24_324a: {1.0: 1.0, 2.0: 2.0}
MARSAME    ANES V241385x: {-2: 'missing', -1: 'missing', 1: 1.0, 2: 2.0, 3: 2.0, 4: 3.0, 5: 4.0, 6: 4.0, 7: 5.0}
FEFAM      ANES V242279x: {-7: 'missing', -6: 'missing', -5: 'missing', -2: 'missing', -1: 'missing', 1: 1.0, 2: 2.0, 3: 2.0, 4: 'missing', 5: 3.0, 6: 3.0, 7: 4.0}
IMMFATE    ANES V241386: {-9: 'missing', -8: 'missing', -1: 'missing', 1: 3.0, 2: 2.0, 3: 1.0

## 6. Save

In [9]:
os.makedirs(OUT_DIR, exist_ok=True)
OUT_COLS = ["sex", "age", "income", "eth", "education", "vote", "state", "time_window_bin", "belief_value",
            "question_id", "benchmark", "wave", "wording_source", "confidence_low", "state_abbr",
            "respondent_id", "interview_date", "post_interview_date"]  # first 11 = the biweekly columns

for survey, survey_parts in parts.items():
    long = pd.concat(survey_parts, ignore_index=True)[OUT_COLS]
    long["belief_value"] = long["belief_value"].astype(int)
    path = f"{OUT_DIR}/basic_{survey.lower()}_target_df_7d.csv"
    long.to_csv(path, index=False)
    print(f"Saved {len(long):,} rows ({long['question_id'].nunique()} questions) to {path}")

map_out = target_map.assign(map=target_map["map"].map(json.dumps), llm_map=target_map["llm_map"].map(json.dumps))
map_out.to_csv(f"{OUT_DIR}/target_map_7d.csv", index=False)
print(f"Saved {len(map_out)} targets to {OUT_DIR}/target_map_7d.csv")

Saved 144,195 rows (30 questions) to ../../Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/target_df_7d/basic_anes_target_df_7d.csv


Saved 931,331 rows (17 questions) to ../../Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/target_df_7d/basic_cces_target_df_7d.csv
Saved 35,174 rows (21 questions) to ../../Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/target_df_7d/basic_gss_target_df_7d.csv
Saved 68 targets to ../../Dropbox/is_culture_king/is_culture_king_Yuyue/generated_data/target_df_7d/target_map_7d.csv
